In [ ]:
%%file producer.py

from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

shops = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
categories = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(shops),
        'category': random.choice(categories),
        'timestamp': datetime.now().isoformat(),
    }

for i in range(10):
    tx = generate_transaction()
    producer.send('transactions', tx)
    print(tx)
    time.sleep(1)

producer.flush()

In [ ]:
import os
import pyspark

# Auto-detect Spark version → correct connector
spark_version = pyspark.__version__
print(f"Detected PySpark version: {spark_version}")

if spark_version.startswith("4"):
    KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2"
else:
    # Spark 3.x
    KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"

print(f"Using connector:         {KAFKA_PACKAGE}")

# Must be set BEFORE SparkSession.builder
os.environ['PYSPARK_SUBMIT_ARGS'] = f'--packages {KAFKA_PACKAGE} pyspark-shell'

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .config("spark.jars.packages", KAFKA_PACKAGE)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — ready")

In [ ]:
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .option("startingOffsets", "earliest")
    .load()
)

kafka_raw.printSchema()

In [ ]:
from pyspark.sql.functions import col

batch_counter = {"n": 0}

def peek_raw(df, batch_id):
    batch_counter["n"] += 1
    print(f"--- Batch {batch_id}: {df.count()} messages ---")
    df.select(
        "topic",
        "partition",
        "offset",
        "timestamp",
        col("key").cast("string").alias("key"),
        col("value").cast("string").alias("value"),   # bytes → UTF-8 text
    ).show(5, truncate=100)
    if batch_counter["n"] >= 2:
        raise Exception("stop")

q = (
    kafka_raw.writeStream
    .foreachBatch(peek_raw)
    .option("checkpointLocation", "/tmp/chk_lab4_peek")
    .start()
)
try:
    q.awaitTermination()
except:
    q.stop()

In [ ]:
# Step 1: binary → string (raw JSON as text)
step1 = kafka_raw.select(
    col("offset"),
    col("partition"),
    col("value").cast("string").alias("raw_json"),
)

batch_counter["n"] = 0

def show_step1(df, batch_id):
    batch_counter["n"] += 1
    print(f"--- Batch {batch_id} ---")
    df.show(3, truncate=120)
    if batch_counter["n"] >= 2:
        raise Exception("stop")

q = step1.writeStream.foreachBatch(show_step1) \
         .option("checkpointLocation", "/tmp/chk_lab4_step1").start()
try:
    q.awaitTermination()
except:
    q.stop()

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import from_json

tx_schema = StructType([
    StructField("tx_id",     StringType()),
    StructField("user_id",   StringType()),
    StructField("amount",    DoubleType()),
    StructField("store",     StringType()),
    StructField("category",  StringType()),
    StructField("timestamp", StringType()),
])

# Step 2: string → struct (single 'tx' column containing all fields)
step2 = kafka_raw.select(
    from_json(col("value").cast("string"), tx_schema).alias("tx")
)

batch_counter["n"] = 0

def show_step2(df, batch_id):
    batch_counter["n"] += 1
    print(f"--- Batch {batch_id} — schema after from_json ---")
    df.printSchema()
    df.show(3, truncate=False)
    if batch_counter["n"] >= 1:
        raise Exception("stop")

q = step2.writeStream.foreachBatch(show_step2) \
         .option("checkpointLocation", "/tmp/chk_lab4_step2").start()
try:
    q.awaitTermination()
except:
    q.stop()

In [ ]:
from pyspark.sql.functions import to_timestamp

# Step 3: struct → flat columns + timestamp conversion
df = (
    kafka_raw
    .select(from_json(col("value").cast("string"), tx_schema).alias("tx"))
    .select("tx.*")
    .withColumn("timestamp", to_timestamp("timestamp", "yyyy-MM-dd HH:mm:ss"))
)

print("Final schema:")
df.printSchema()

batch_counter["n"] = 0

def show_parsed(df, batch_id):
    batch_counter["n"] += 1
    print(f"--- Batch {batch_id} ---")
    df.show(5, truncate=False)
    if batch_counter["n"] >= 2:
        raise Exception("stop")

q = df.writeStream.foreachBatch(show_parsed) \
      .option("checkpointLocation", "/tmp/chk_lab4_parsed").start()
try:
    q.awaitTermination()
except:
    q.stop()

In [ ]:
from pyspark.sql.functions import window, count, sum as _sum, round as _round

windowed = (
    df
    .withWatermark("timestamp", "30 seconds")
    .groupBy(window("timestamp", "1 minute"), "store")
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_amount"),
    )
)

batch_counter["n"] = 0

def show_window(df, batch_id):
    batch_counter["n"] += 1
    print(f"\n=== Batch {batch_id} ===")
    (
        df.select(
            col("window.start").alias("from"),
            col("window.end").alias("to"),
            "store", "tx_count", "total_amount",
        )
        .orderBy("from", "store")
        .show(truncate=False)
    )
    if batch_counter["n"] >= 5:
        raise Exception("stop")

q = (
    windowed.writeStream
    .outputMode("append")
    .foreachBatch(show_window)
    .option("checkpointLocation", "/tmp/chk_lab4_windows")
    .start()
)
try:
    q.awaitTermination()
except:
    q.stop()

In [ ]:
from pyspark.sql.functions import to_json, struct, lit

alerts = (
    df
    .filter(col("amount") > 3000)
    .select(
        to_json(
            struct(
                "tx_id", "user_id", "amount", "store", "category",
                col("timestamp").cast("string"),
                lit("HIGH").alias("alert_level"),
            )
        ).alias("value")    # Kafka requires a 'value' column
    )
)

alert_query = (
    alerts.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("topic", "alerts")
    .option("checkpointLocation", "/tmp/chk_lab4_alerts")
    .outputMode("append")
    .start()
)
print("Alert stream started. Stop manually: alert_query.stop()")

In [ ]:
##HW1: 
from pyspark.sql.functions import window, count, sum as _sum, round as _round

sliding_window = (
    df
    .withWatermark("timestamp", "30 seconds")
    .groupBy(
        window("timestamp", "2 minutes", "1 minute"),
        "store"
    )
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_amount")
    )
)

query = (
    sliding_window.writeStream
    .outputMode("append")
    .foreachBatch(show_window)
    .option(
        "checkpointLocation",
        "/tmp/chk_hw_sliding"
    )
    .start()
)

query.awaitTermination()

In [ ]:
##HW2:
from pyspark.sql.functions import lit

alerts = (
    df
    .filter(col("amount") > 3000)
    .withColumn(
        "ratio",
        col("amount") / 400.0
    )
    .select(
        to_json(
            struct(
                "tx_id",
                "user_id",
                "amount",
                "ratio",
                "store",
                "category",
                col("timestamp").cast("string"),
                lit("HIGH").alias("alert_level"),
            )
        ).alias("value")
    )
)